In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pypulseq as pp
import MRzeroCore as mr0
import sys
sys.path.insert(0, "..")
from mri import load_brain_phantom, default_system

In [ ]:
def main(
    plot: bool = False,
    test_report: bool = False,
    write_seq: bool = False,
    fov: float | tuple[float, float, float] = 256e-3,
    n_x: int = 128,
    n_y: int = 128,
    n_z: int = 128,
    flip_angle_deg: float = 10,
    slice_thickness: float = 3e-3,
    tr: float = 12e-3,
    te: float = 5e-3,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    fov_x, fov_y, fov_z = (fov, fov, fov) if isinstance(fov, (int, float)) else fov
    delta_kx = 1 / fov_x
    delta_ky = 1 / fov_y
    delta_kz = 1 / fov_z
    rf_spoiling_inc = 0

    rf, gz, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(flip_angle_deg),
        duration=3e-3,
        slice_thickness=slice_thickness,
        apodization=0.42,
        time_bw_product=4,
        system=system,
        return_gz=True,
        delay=system.rf_dead_time,
        use='excitation',
    )

    gx = pp.make_trapezoid(channel='x', flat_area=n_x * delta_kx, flat_time=3.2e-3, system=system)
    adc = pp.make_adc(num_samples=n_x, duration=gx.flat_time, delay=gx.rise_time, system=system)
    gx_pre = pp.make_trapezoid(channel='x', area=-gx.area / 2, duration=1e-3, system=system)
    gz_reph = pp.make_trapezoid(channel='z', area=-gz.area / 2, duration=1e-3, system=system)
    phase_areas = (np.arange(n_y) - n_y / 2) * delta_ky
    phase_areas_z = (np.arange(n_z) - n_z / 2) * delta_kz

    gx_spoil = pp.make_trapezoid(channel='x', area=2 * n_x * delta_kx, system=system)
    gy_spoil = pp.make_trapezoid(channel='y', area=2 * n_y * delta_ky, system=system)
    gz_spoil = pp.make_trapezoid(channel='z', area=4 / fov_z, system=system)

    EPS: float = 1e-12

    te_delay = (
        te
        - (pp.calc_duration(gz, rf) - pp.calc_rf_center(rf)[0] - rf.delay)
        - pp.calc_duration(gx_pre)
        - pp.calc_duration(gx) / 2
        - EPS
    )
    te_delay = np.ceil(te_delay / seq.grad_raster_time) * seq.grad_raster_time

    tr_delay = tr - pp.calc_duration(gz, rf) - pp.calc_duration(gx_pre) - pp.calc_duration(gx) - te_delay
    tr_delay = np.ceil(tr_delay / seq.grad_raster_time) * seq.grad_raster_time

    assert np.all(te_delay >= 0)
    assert np.all(tr_delay >= pp.calc_duration(gx_spoil, gy_spoil, gz_spoil))

    rf_phase = 0
    rf_inc = 0

    for i_z in range(n_z):  
        for i_phase in range(n_y):  
            rf.phase_offset = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_inc = divmod(rf_inc + rf_spoiling_inc, 360.0)[1]
            rf_phase = divmod(rf_phase + rf_inc, 360.0)[1]

            seq.add_block(rf, gz)

            gy_pre = pp.make_trapezoid(
                channel='y',
                area=phase_areas[i_phase],
                duration=pp.calc_duration(gx_pre),
                system=system,
            )

            gz_pre = pp.make_trapezoid(
                channel='z',
                area=phase_areas_z[i_z] - gz.area / 2,
                duration=pp.calc_duration(gx_pre),
                system=system,
            )
            seq.add_block(gx_pre, gy_pre, gz_pre)

            seq.add_block(pp.make_delay(te_delay))
            seq.add_block(gx, adc)

            gy_rewind = pp.make_trapezoid(channel='y', area=-gy_pre.area, system=system)

            gz_combined_spoil = pp.make_trapezoid(
                channel='z',
                area=gz_spoil.area - gz_pre.area,
                system=system
            )

            seq.add_block(pp.make_delay(tr_delay), gx_spoil, gy_rewind, gz_combined_spoil)

    ok, error_report = seq.check_timing()
    if ok:
        print('Timing check passed successfully')
    else:
        print('Timing check failed. Error listing follows:')
        [print(e) for e in error_report]

    if test_report:
        print(seq.test_report())

    if plot:
        seq.plot(time_range=(0.0, tr))

    seq.set_definition('FOV', [fov_x, fov_y, fov_z])
    seq.set_definition('Name', 'gre')

    if write_seq:
        seq.write("3D GRE.seq")

    return seq

if __name__ == '__main__':
    main(plot=True, write_seq=True)

In [ ]:
n_x: int = 128
n_y: int = 128
n_z: int = 128

device = torch.device("cuda")
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n_z, n_y, n_x),
    device=device,
    permute_order=(2, 1, 0),
)

signal,_ = mr0.util.simulate("3D GRE.seq", phantom)

In [ ]:
kspace = signal.squeeze().numpy().reshape(n_z, n_y, n_x)

img = np.fft.ifftn(np.fft.ifftshift(kspace, axes=(0,1,2)), axes=(0,1,2))
img = np.fft.fftshift(img, axes=(0,1,2))

center_slice = n_z // 2

plt.figure(figsize=(6, 6))
plt.imshow(np.abs(img[center_slice, :, :]), cmap='gray')
plt.title(f"Reconstructed 3D GRE - Axial Slice {center_slice}")
plt.axis('off')
plt.show()